# Debugging inside a notebook: `%debug`, `%pdb`, `breakpoint()` and the pdb commands

When a cell raises, the traceback tells you *where*; the debugger lets you look at
*what the variables were* at that moment, without adding prints and re-running.

This notebook shows every command with real output. The output comes from a scripted
pdb (defined in the next cell) that types the commands for you. In your own session you
type them at the `(Pdb)` prompt. Cells marked **run this yourself** are interactive and
are left commented out.

**What's in here**
1. Read the traceback first; `%xmode Verbose` shows the local variables
2. `%debug`: post-mortem on the last error (`w`, `l`, `ll`, `p`, `pp`, `u`, `d`, `a`, `q`)
3. `%pdb on`: open the debugger automatically on every error
4. `breakpoint()`: stop inside running code and step (`n`, `s`, `r`, `c`, `until`, `b`, `display`)
5. Command names vs variable names (`p n`, `!n = 5`)
6. pandas: finding the row that broke `apply`, the group that broke `groupby`
7. The same without pdb: walking the traceback frames in code
8. Turn silent problems into errors (warnings, chained assignment, float errors)
9. PyCharm notes
10. Quick reference

In [1]:
import io
import pdb
import sys
import traceback

class ScriptedPdb(pdb.Pdb):
    """A pdb that reads its commands from a list instead of the keyboard.
    Used only so this notebook can show real (Pdb) output. In your own session
    you type the same commands at the (Pdb) prompt."""
    def __init__(self, commands):
        super().__init__(stdin=io.StringIO("\n".join(commands) + "\n"), stdout=sys.stdout)
        self.use_rawinput = False
        self.prompt = "(Pdb) "
        if hasattr(self, "set_colors"):      # IPython's pdb subclass adds colours; keep output plain
            self.set_colors("NoColor")
    def precmd(self, line):
        print(line)              # echo the command so you can see what was typed
        return line

def post_mortem(commands, tb):
    """Like typing %debug and then these commands."""
    dbg = ScriptedPdb(commands)
    dbg.reset()
    dbg.interaction(None, tb)

def step_through(commands, func, *args):
    """Like putting breakpoint() at the top of func and then typing these commands."""
    dbg = ScriptedPdb(commands)
    try:
        return dbg.runcall(func, *args)
    except pdb.bdb.BdbQuit:
        print("(debugger quit)")

## 1. Read the traceback first

Three small functions; the innermost one divides by zero for some inputs.

In [2]:
def clean(x):
    y = x * 2
    return 10 / (y - 4)          # fails when y == 4, i.e. x == 2

def process(a):
    b = a + 1
    return clean(b)

def run_all(values):
    results = []
    for v in values:
        results.append(process(v))
    return results

Run it. The traceback lists the frames top to bottom: the cell, `run_all`, `process`,
`clean`. **Read it from the bottom**: the last line is the error, the frame above it is
where it happened, and each frame shows the line that was executing.

In [3]:
run_all([0, 1, 5])

ZeroDivisionError: division by zero

`%xmode Verbose` makes IPython print the **local variables of every frame** inside the
traceback. Often that alone answers "which input broke it?" — here `x = 2`, `a = 1`,
`v = 1`.

In [4]:
%xmode Verbose
run_all([0, 1, 5])

Exception reporting mode: Verbose


ZeroDivisionError: division by zero

In [5]:
%xmode Context

Exception reporting mode: Context


## 2. `%debug`: post-mortem on the last error

After a cell fails, run `%debug` in a new cell. You land at the `(Pdb)` prompt **inside
the frame that raised**, with all its variables alive. Type commands, then `q` to leave.

**Run this yourself**: uncomment the next cell (the error from section 1 is the last one).

In [6]:
# %debug

Below is what that session looks like. Commands used:

- `w` (where): the stack, `>` marks the current frame
- `l` (list): code around the current line; `ll` (longlist): the whole current function
- `p x` (print): value of `x`; `pp` pretty-prints; `p x, y` prints several
- `u` (up): move one frame up the stack, towards the caller; `d` (down): back
- `a` (args): the arguments of the current function
- `q` (quit)

In [7]:
try:
    run_all([0, 1, 5])
except ZeroDivisionError:
    tb = sys.exc_info()[2]        # this is what %debug uses (sys.last_traceback)

post_mortem(["w", "l", "p x", "p y", "p x, y", "ll", "q"], tb)

> /tmp/ipykernel_199118/3590463174.py(3)clean()
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

w
  /tmp/ipykernel_199118/3578418963.py(2)<module>()
      1 try:
----> 2     run_all([0, 1, 5])
      3 except ZeroDivisionError:
      4     tb = sys.exc_info()[2]        # this is what %debug uses (sys.last_traceback)
      5 

  /tmp/ipykernel_199118/3590463174.py(12)run_all()
      9 def run_all(values):
     10     results = []
     11     for v in values:
---> 12         results.append(process(v))
     13     return results

  /tmp/ipykernel_199118/3590463174.py(7)process()
      5 def process(a):
      6     b = a + 1
----> 7     return clean(b)
      8 
      9 def run_all(values):

> /tmp/ipykernel_199118/3590463174.py(3)clean()
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

l
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):
      6     b = a + 1
      7     return clean(b)
      8 
      9 def run_all(values):
     10     results = []
     11     for v in values:

(Pdb) 

p x
2
(Pdb) 

p y
4
(Pdb) 

p x, y
(2, 4)
(Pdb) 

ll
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 

(Pdb) 

q


`x` is 2 and `y` is 4, so `y - 4` is 0. Now move **up** to see what the caller was doing.

In [8]:
post_mortem(["u", "a", "p b", "u", "p v", "p values", "p results", "d", "d", "p x", "q"], tb)

> /tmp/ipykernel_199118/3590463174.py(3)clean()
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

u
> /tmp/ipykernel_199118/3590463174.py(7)process()
      5 def process(a):
      6     b = a + 1
----> 7     return clean(b)
      8 
      9 def run_all(values):

(Pdb) 

a
a = 1
(Pdb) 

p b
2
(Pdb) 

u
> /tmp/ipykernel_199118/3590463174.py(12)run_all()
      9 def run_all(values):
     10     results = []
     11     for v in values:
---> 12         results.append(process(v))
     13     return results

(Pdb) 

...

p results
[-5.0]
(Pdb) 

d
> /tmp/ipykernel_199118/3590463174.py(7)process()
      5 def process(a):
      6     b = a + 1
----> 7     return clean(b)
      8 
      9 def run_all(values):

(Pdb) 

d
> /tmp/ipykernel_199118/3590463174.py(3)clean()
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

p x
2
(Pdb) 

q


Each `u` goes one frame towards the caller: `process` (where `a = 1`, `b = 2`), then
`run_all` (where `v = 1`, and `results` shows that `v = 0` had already succeeded).
`d` goes back down.

**Interview check:** "the error is inside pandas internals, ten frames deep — how do you
get to your own code?" `u` repeatedly until the file name in the `>` line is your cell,
or `w` first to count how many frames up that is.

## 3. `%pdb on`: open the debugger automatically

With `%pdb on`, every uncaught exception drops you straight into the post-mortem prompt
(no need to type `%debug`). `%pdb off` turns it back off, `%pdb` alone toggles.

**Run this yourself**: `%pdb on`, run the failing cell from section 1, look around, `q`,
then `%pdb off`. Leaving it on is fine while you debug; turn it off before a
run-all, otherwise the first error stops the notebook waiting for input.

In [9]:
%pdb on
%pdb off

Automatic pdb calling has been turned ON
Automatic pdb calling has been turned OFF


## 4. `breakpoint()`: stop inside running code and step

Put `breakpoint()` (or `import pdb; pdb.set_trace()`) on the line where you want to
stop. When execution reaches it you get the `(Pdb)` prompt *before* the error happens.
Stepping commands:

- `n` (next): run the current line, stay in this function
- `s` (step): like `n`, but if the line calls a function, go **into** it
- `r` (return): run until the current function returns
- `c` (continue): run until the next breakpoint or the end
- `until <line>`: run until that line number (handy to get out of a loop)
- `b <line>` / `b <func>`: set a breakpoint; `b <line>, <cond>` only when cond is true; `cl` clears
- `display <expr>`: print the expression every time execution stops

**Run this yourself**: uncomment `breakpoint()` in the cell below, run the next cell,
and type `n`, `n`, `p b`, `s`, `p x`, `c`.

In [10]:
def process_bp(a):
    # breakpoint()            # <- uncomment to stop here in your own session
    b = a + 1
    return clean(b)

In [11]:
# process_bp(5)

Scripted version of that session. `s` on `return clean(b)` steps **into** `clean`.

In [12]:
step_through(["n", "p b", "s", "p x", "n", "n", "p y", "r", "c"], process_bp, 5)

> /tmp/ipykernel_199118/1113896328.py(3)process_bp()
      1 def process_bp(a):
      2     # breakpoint()            # <- uncomment to stop here in your own session
----> 3     b = a + 1
      4     return clean(b)

(Pdb) 

n
> /tmp/ipykernel_199118/1113896328.py(4)process_bp()
      1 def process_bp(a):
      2     # breakpoint()            # <- uncomment to stop here in your own session
      3     b = a + 1
----> 4     return clean(b)

(Pdb) 

p b
6
(Pdb) 

s
--Call--
> /tmp/ipykernel_199118/3590463174.py(1)clean()
----> 1 def clean(x):
      2     y = x * 2
      3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

p x
6
(Pdb) 

...

n
> /tmp/ipykernel_199118/3590463174.py(3)clean()
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

p y
12
(Pdb) 

r
--Return--
1.25
> /tmp/ipykernel_199118/3590463174.py(3)clean()
      1 def clean(x):
      2     y = x * 2
----> 3     return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
      4 
      5 def process(a):

(Pdb) 

c


1.25

Read it line by line:
- the first stop is *before* `b = a + 1` runs; `n` executes it, so `p b` then shows 6.
- `s` at `return clean(b)` steps **into** `clean` and stops at its `def` line (`--Call--`);
  `p x` already shows 6. The first `n` moves to `y = x * 2`, the second runs it, so `p y` shows 12.
- `r` runs `clean` to its return and shows the value (`->1.25`); `c` finishes.

A loop: use `until` or a conditional breakpoint instead of pressing `n` fifty times.

In [13]:
def total_up(values):
    total = 0
    for i, v in enumerate(values):
        total = total + v
    return total

step_through(["n", "display total", "n", "n", "n", "n", "n", "p i, v, total", "c"], total_up, [10, 20, 30, 40])

> /tmp/ipykernel_199118/3949364042.py(2)total_up()
      1 def total_up(values):
----> 2     total = 0
      3     for i, v in enumerate(values):
      4         total = total + v
      5     return total

(Pdb) 

n
> /tmp/ipykernel_199118/3949364042.py(3)total_up()
      1 def total_up(values):
      2     total = 0
----> 3     for i, v in enumerate(values):
      4         total = total + v
      5     return total

(Pdb) 

display total
display total: 0
(Pdb) 

n
> /tmp/ipykernel_199118/3949364042.py(4)total_up()
      2     total = 0
      3     for i, v in enumerate(values):
----> 4         total = total + v
      5     return total
      6 

(Pdb) 

n
> /tmp/ipykernel_199118/3949364042.py(3)total_up()
      1 def total_up(values):
      2     total = 0
----> 3     for i, v in enumerate(values):
      4         total = total + v
      5     return total

display total: 10  [old: 0]
(Pdb) 

...

n
> /tmp/ipykernel_199118/3949364042.py(3)total_up()
      1 def total_up(values):
      2     total = 0
----> 3     for i, v in enumerate(values):
      4         total = total + v
      5     return total

display total: 30  [old: 10]
(Pdb) 

n
> /tmp/ipykernel_199118/3949364042.py(4)total_up()
      2     total = 0
      3     for i, v in enumerate(values):
----> 4         total = total + v
      5     return total
      6 

(Pdb) 

p i, v, total
(2, 30, 30)
(Pdb) 

c


100

The first `n` runs `total = 0`; from then on `display total` re-prints `total` at every
stop where it changed, so you can watch it grow (10, 30, 60) without typing `p total`
each time.

Conditional breakpoint: stop only when the condition holds. `b` takes a line number of
the current file; in a notebook `ll` shows the numbers to use. Here the condition is on
the loop variable.

In [14]:
def find_bad(values):
    out = []
    for v in values:
        out.append(process(v))       # process(1) will fail
    return out

# stop inside the loop only when v == 1, then look before the crash
step_through(["ll", "b 4, v == 1", "c", "p v", "p out", "q"], find_bad, [0, 3, 1, 5])

> /tmp/ipykernel_199118/2978617382.py(2)find_bad()
      1 def find_bad(values):
----> 2     out = []
      3     for v in values:
      4         out.append(process(v))       # process(1) will fail
      5     return out

(Pdb) 

ll
      1 def find_bad(values):
----> 2     out = []
      3     for v in values:
      4         out.append(process(v))       # process(1) will fail
      5     return out
      6 

(Pdb) 

b 4, v == 1
Breakpoint 1 at /tmp/ipykernel_199118/2978617382.py:4
(Pdb) 

c
> /tmp/ipykernel_199118/2978617382.py(4)find_bad()
      2     out = []
      3     for v in values:
1---> 4         out.append(process(v))       # process(1) will fail
      5     return out
      6 

(Pdb) 

p v
1
(Pdb) 

p out
[-5.0, 2.5]
(Pdb) 

q


`b 4, v == 1` sets a breakpoint on line 4 of the function (the `out.append` line, as
`ll` showed) that only fires when `v == 1`. `c` runs there; `p out` shows what had been
accumulated before the failing input.

## 5. Command names vs variable names

pdb commands are single letters, so `n`, `l`, `p`, `c`, `s`, `d`, `u`, `a`, `w`, `r`, `b`
clash with variables of the same name. `p n` prints the variable `n`; `n` alone runs the
next line. To run any Python statement, prefix it with `!`.

In [15]:
def with_short_names(n, l):
    c = n * 2
    return c + l

step_through(["p n", "p l", "n", "p c", "!c = 100", "p c", "!import pandas as pd; print(pd.__version__)", "c"], with_short_names, 3, 4)

> /tmp/ipykernel_199118/205571549.py(2)with_short_names()
      1 def with_short_names(n, l):
----> 2     c = n * 2
      3     return c + l
      4 
      5 step_through(["p n", "p l", "n", "p c", "!c = 100", "p c", "!import pandas as pd; print(pd.__version__)", "c"], with_short_names, 3, 4)

(Pdb) 

p n
3
(Pdb) 

p l
4
(Pdb) 

n
> /tmp/ipykernel_199118/205571549.py(3)with_short_names()
      1 def with_short_names(n, l):
      2     c = n * 2
----> 3     return c + l
      4 
      5 step_through(["p n", "p l", "n", "p c", "!c = 100", "p c", "!import pandas as pd; print(pd.__version__)", "c"], with_short_names, 3, 4)

(Pdb) 

p c
6
(Pdb) 

...

p c
100
(Pdb) 

!import pandas as pd; print(pd.__version__)


2.3.3
(Pdb) 

c


104

`!c = 100` changed the variable, and the function then returned 104 instead of 10. That
is a quick way to test "would it work with this value?" without editing and re-running.

## 6. pandas: which row broke `apply`, which group broke `groupby`

An error inside `df.apply(f, axis=1)` is raised deep inside pandas. The traceback ends
in your function `f`; `p row` there shows the offending row and `p row.name` its label.

In [16]:
import numpy as np
import pandas as pd

df = pd.DataFrame({"meter": ["A", "B", "C", "D"], "kwh": [1.0, 2.0, np.nan, 4.0], "days": [1, 2, 3, 0]})

def per_day(row):
    if row["days"] == 0:
        raise ValueError("days is zero")
    return row["kwh"] / row["days"]

try:
    df.apply(per_day, axis=1)
except ValueError:
    tb = sys.exc_info()[2]

post_mortem(["w", "p row", "p row.name", "q"], tb)

> /tmp/ipykernel_199118/1352075980.py(8)per_day()
      6 def per_day(row):
      7     if row["days"] == 0:
----> 8         raise ValueError("days is zero")
      9     return row["kwh"] / row["days"]
     10 

(Pdb) 

w
  /tmp/ipykernel_199118/1352075980.py(12)<module>()
     10 
     11 try:
---> 12     df.apply(per_day, axis=1)
     13 except ValueError:
     14     tb = sys.exc_info()[2]

  /home/dim/.local/lib/python3.9/site-packages/pandas/core/frame.py(10401)apply()
  10399             kwargs=kwargs,
  10400         )
> 10401         return op.apply().__finalize__(self, method="apply")
  10402 
  10403     def map(

  /home/dim/.local/lib/python3.9/site-packages/pandas/core/apply.py(916)apply()
    914             return self.apply_raw(engine=self.engine, engine_kwargs=self.engine_kwargs)
    915 
--> 916         return self.apply_standard()
    917 
    918     def agg(self):

  /home/dim/.local/lib/python3.9/site-packages/pandas/core/apply.py(1063)apply_standard()
   1061     def apply_standard(self):
   1062         if self.engine == "python":
-> 1063             results, res_index = self.apply_series_generator()
   1064         else:
   1065             results, res_index = self.apply_seri

p row
meter      D
kwh      4.0
days       0
Name: 3, dtype: object
(Pdb) 

p row.name
3
(Pdb) 

q


`w` shows several pandas frames between your cell and `per_day`; you do not need to read
them. The `>` frame is `per_day`, and `p row` shows meter `D` with `days = 0`.

The same for a `groupby(...).apply`: `p group.name` (or the key column) identifies the group.

In [17]:
def check_group(g):
    if g["kwh"].isna().any():
        raise ValueError("NaN in group")
    return g["kwh"].sum()

try:
    df.groupby("meter").apply(check_group)
except ValueError:
    tb = sys.exc_info()[2]

post_mortem(["p g", "p g.name", "q"], tb)

> /tmp/ipykernel_199118/221958355.py(3)check_group()
      1 def check_group(g):
      2     if g["kwh"].isna().any():
----> 3         raise ValueError("NaN in group")
      4     return g["kwh"].sum()
      5 

(Pdb) 

p g
  meter  kwh  days
2     C  NaN     3
(Pdb) 

p g.name
'C'
(Pdb) 

q


Useful pdb one-liners for DataFrames while you are in a frame:

- `p df.shape`, `p df.dtypes`, `p df.head()`, `p df.index[:5]`
- `pp df.isna().sum().to_dict()`
- `p df.loc[df["kwh"].isna()]`
- `!df.to_pickle("/tmp/debug.pkl")` to save the object and inspect it in another cell

## 7. The same without pdb: walk the traceback in code

`%debug` is interactive. Sometimes you want the same information printed once, or you
are in a context without a prompt. A traceback is a linked list of frames; each frame
has `f_locals`.

In [18]:
try:
    run_all([0, 1, 5])
except ZeroDivisionError as e:
    exc, tb = e, sys.exc_info()[2]

frame = tb.tb_next            # skip the cell's own frame (its locals are the whole notebook namespace)
while frame is not None:
    f = frame.tb_frame
    local_vars = {k: v for k, v in f.f_locals.items() if not k.startswith("_")}
    print(f"{f.f_code.co_name:10s} line {frame.tb_lineno:3d}  locals: {local_vars}")
    frame = frame.tb_next

run_all    line  12  locals: {'values': [0, 1, 5], 'results': [-5.0], 'v': 1}
process    line   7  locals: {'a': 1, 'b': 2}
clean      line   3  locals: {'x': 2, 'y': 4}


That is exactly what `%xmode Verbose` prints, and what you would see with `%debug` plus
`w`/`u`/`p`. `traceback.extract_tb` gives the same stack as a list of records.

In [19]:
for rec in traceback.extract_tb(tb):
    print(rec.name, "->", rec.line)
print("raised in:", traceback.extract_tb(tb)[-1].name)
print("exception :", type(exc).__name__, "-", exc)

<module> -> run_all([0, 1, 5])
run_all -> results.append(process(v))
process -> return clean(b)
clean -> return 10 / (y - 4)          # fails when y == 4, i.e. x == 2
raised in: clean
exception : ZeroDivisionError - division by zero


After a cell fails, IPython also keeps the last error in `sys.last_type`, `sys.last_value`
and `sys.last_traceback`; `%debug` reads `sys.last_traceback`.

## 8. Turn silent problems into errors

Some of the worst notebook bugs never raise. Make them raise while you debug, then you
can `%debug` them like anything else.

In [20]:
import warnings

df2 = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

pd.options.mode.chained_assignment = "raise"     # default is "warn"
try:
    df2[df2["a"] > 1]["b"] = 0                     # the silent no-op becomes an error
except Exception as e:
    print(type(e).__name__, "-", str(e)[:60])
pd.options.mode.chained_assignment = "warn"

SettingWithCopyError - 
A value is trying to be set on a copy of a slice from a Dat


In [21]:
with warnings.catch_warnings():
    warnings.simplefilter("error")                 # any warning becomes an exception
    try:
        pd.to_datetime(["01/02/2023", "13/02/2023"])   # day-first ambiguity warning
    except Exception as e:
        print(type(e).__name__, "-", str(e)[:70])

ValueError - time data "13/02/2023" doesn't match format "%m/%d/%Y", at position 1.


In [22]:
with np.errstate(all="raise"):                     # 1/0 in numpy normally gives inf + a warning
    try:
        np.array([1.0]) / np.array([0.0])
    except FloatingPointError as e:
        print("FloatingPointError -", e)

FloatingPointError - divide by zero encountered in divide


## 9. PyCharm notes

- In a PyCharm notebook, `%debug` and `breakpoint()` open a text `(Pdb)` prompt in the
  cell output; type commands there exactly as above.
- PyCharm also has its own graphical debugger for notebooks: set a breakpoint in the
  gutter of a cell and use **Debug Cell** (the bug icon next to Run). You get the usual
  variables panel, step over/into (F8/F7) and evaluate-expression. Same ideas, different keys.
- Outside notebooks, `python -m pdb script.py` starts the script under pdb, and
  `python -m pdb -c continue script.py` runs until the first error.

## 10. Quick reference

| Command | Meaning |
|---|---|
| `%debug` | post-mortem prompt for the last error |
| `%pdb on` / `off` | open the prompt automatically on every error |
| `%xmode Verbose` | print local variables inside tracebacks |
| `breakpoint()` | stop here when the code runs |
| `w` | where: show the stack, `>` is the current frame |
| `l` / `ll` | list code around the line / the whole function |
| `p x` / `pp x` | print / pretty-print `x` |
| `a` | arguments of the current function |
| `u` / `d` | up towards the caller / down |
| `n` | next line (step over calls) |
| `s` | step into the call on this line |
| `r` | run until this function returns |
| `until N` | run until line N |
| `b N` / `b N, cond` / `cl` | breakpoint at line N / conditional / clear |
| `display expr` | show `expr` at every stop |
| `!stmt` | run a Python statement (e.g. `!x = 5`) |
| `c` | continue |
| `q` | quit the debugger |
| `h` | help; `h n` help on one command |

Habit to build: **traceback bottom-up → `%debug` → `w` → `p` the inputs of the failing
frame → `u` until you reach your own code → `p` what you passed in.**